# DLP NPPE-1 (T22026): Legal & Policy Document Classification — LoRA (PEFT) Version

## Task

The objective of NPPE 1 is to build an NLP model that automatically classifies legal and policy documents into the correct category using Parameter Efficient Fine Tuning (PEFT) techniques.

## Overview

In this challenge, we develop a Transformer-based text classification model for legal and policy documents. The goal is to build an accurate and efficient document classifier by applying the concepts learned during the course.

The competition focuses on practical aspects of Transformer fine-tuning, including:

- Exploring different pretrained Transformer models for document classification
- Building an effective preprocessing and tokenization pipeline
- Applying PEFT methods such as LoRA or QLoRA
- Optimizing hyperparameters such as learning rate, batch size, and training schedule
- Evaluating model performance using the Accuracy Score

Since the documents vary significantly in length and writing style, thoughtful preprocessing and model optimization play an important role in achieving strong performance.

**Dataset:** `train.csv` (id, text, label), `test.csv` (id, text), 30 target classes. Documents are EU legal/policy texts (Council Decisions, Commission Regulations, Directives, etc.), varying widely in length.

**Evaluation Metric:** Accuracy Score.

**Submission Format:** CSV with columns `ID`, `label`.

---

## This Notebook: LoRA (PEFT) Approach

This notebook implements **LoRA (Low-Rank Adaptation)** — a Parameter-Efficient Fine-Tuning (PEFT) technique that freezes the pretrained model's weights and injects small trainable low-rank adapter matrices into the attention layers. Only these adapters (plus the classification head) are trained, which is much faster and more memory-efficient than full fine-tuning.

**Result achieved: 0.708 (leaderboard).**

**Note:** a separate notebook implements **Full Fine-Tuning (FFT)** of the same task on a domain-matched legal-BERT model, which reached a higher accuracy (~0.73-0.75). That version is intentionally excluded here — this notebook documents the LoRA/PEFT approach specifically, as required by the assignment's PEFT requirement.

| Aspect | This notebook (LoRA/PEFT) | Alternative (FFT, separate notebook) |
|---|---|---|
| Base model | `roberta-base` | `nlpaueb/legal-bert-base-uncased` |
| Trainable params | ~1.2M (~1%) | ~110M (100%) |
| Learning rate | 2e-4 (LoRA needs higher LR) | 2e-5 (FFT needs much gentler LR) |
| Result | 0.708 | ~0.73-0.75 |
| Training time | ~1.5 hrs | ~1.5-2 hrs |


# 1. Setup & Imports

This cell installs `peft` (the library that implements LoRA and other PEFT methods) alongside pinned `transformers`/`accelerate`/`datasets` versions, then imports everything needed.

Environment variables are set to: force single-GPU usage (Kaggle's dual-T4 setup otherwise triggers a slow naive `DataParallel` path), reduce CUDA memory fragmentation, and silence a harmless tokenizer fork warning.


In [1]:
# Install required packages
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "peft==0.11.1", "transformers==4.41.2", "accelerate==0.30.1",
                 "datasets==2.19.1"])
print("[INFO] Packages installed.")

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # force single GPU - avoids slow naive DataParallel on Kaggle T4x2
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduces OOM from memory fragmentation
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # silences a tokenizer fork warning

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print(f"[INFO] torch version: {torch.__version__}")
print(f"[INFO] CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] Num GPUs: {torch.cuda.device_count()}")
else:
    print("[WARNING] No GPU found! Go to Settings -> Accelerator -> GPU T4 x2")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


[INFO] Packages installed.
[INFO] torch version: 2.10.0+cu128
[INFO] CUDA available: True
[INFO] GPU: Tesla T4
[INFO] Num GPUs: 1


**Insight:** Confirm `CUDA available: True` before continuing. `CUDA_VISIBLE_DEVICES="0"` deliberately restricts training to a single GPU — on Kaggle's T4 x2 accelerator, the `Trainer`'s default naive `DataParallel` across both GPUs is *slower* than single-GPU training for a model this size, due to synchronization overhead outweighing the benefit of parallelism.


# 2. Load Competition Data

We load `train.csv`, `test.csv`, and `sample_submission.csv`, and inspect shapes, label distribution, and a word-count summary of the documents — the latter directly informs the tokenizer's `max_length` choice in Section 5.


In [2]:
# Locate input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

DATA_DIR = "/kaggle/input/competitions/dlp-nppe-1-t-22026"
train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

print(f"[INFO] train shape: {train_df.shape}")
print(f"[INFO] test shape: {test_df.shape}")
print(f"[INFO] train columns: {list(train_df.columns)}")
print(train_df.head(2))
print(f"[INFO] label distribution:\n{train_df['label'].value_counts().sort_index()}")

NUM_LABELS = train_df['label'].nunique()
print(f"[INFO] Number of unique labels: {NUM_LABELS}")
print(f"[INFO] Label min/max: {train_df['label'].min()} / {train_df['label'].max()}")

# Quick text length check to decide max_length for tokenizer
train_df['word_count'] = train_df['text'].str.split().str.len()
print(f"[INFO] text word count stats:\n{train_df['word_count'].describe()}")
train_df.drop(columns=['word_count'], inplace=True)


/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv
[INFO] train shape: (50840, 3)
[INFO] test shape: (12710, 2)
[INFO] train columns: ['id', 'text', 'label']
   id                                               text  label
0   0  COUNCIL DECISION\nof 31 January 2011\non the c...      4
1   1  DECISION OF THE EUROPEAN PARLIAMENT AND OF THE...      5
[INFO] label distribution:
label
0     2796
1     5326
2     2066
3     2553
4     1976
5     3515
6     2458
7     7058
8     5938
9     2315
10     778
11    3310
12    2994
13    1376
14     559
15    1056
16     665
17     187
18     190
19    1862
20     241
21      75
22      98
23     202
24      35
25     679
26     390
27      33
28     102
29       7
Name: count, dtype: int64
[INFO] Number of unique labels: 30
[INFO] Label min/max: 0 / 29
[INFO] text word count stats:
count     50840.000000
mean       1186.

**Insight:** 30 label classes (0-29) with a long-tailed distribution — some classes are far more common than others. The word-count summary typically shows most documents fit within a few hundred words, which is why `MAX_LEN=384` (Section 5) is a reasonable truncation budget without needing the full 512-token BERT limit.


# 3. Text Cleaning

Legal documents carry irregular whitespace and line breaks from their original PDF/HTML source. We apply only light cleaning (whitespace collapsing) and deliberately avoid aggressive normalization — legal-specific tokens like section numbers and "Regulation (EC) No." carry classification-relevant signal that heavier cleaning would destroy.


In [3]:
def clean_text(text):
    text = str(text)
    text = " ".join(text.split())  # collapse extra whitespace/newlines
    return text.strip()

train_df['text'] = train_df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)
print("[INFO] Text cleaning done.")


[INFO] Text cleaning done.


**Insight:** Minimal cleaning by design — this preserves document structure that the model can learn to associate with specific categories (e.g. "COUNCIL DECISION" vs. "COMMISSION REGULATION" headers).


# 4. Train/Validation Split

A stratified 90/10 split ensures all 30 classes — including rare ones — are proportionally represented in both the training and validation sets.


In [4]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['label']
)
print(f"[INFO] train_split: {train_split.shape}, val_split: {val_split.shape}")


[INFO] train_split: (45756, 3), val_split: (5084, 3)


**Insight:** ~45.7K rows for training, ~5.1K held out for validation, with class proportions preserved via stratification.


# 5. Model & Tokenizer

**Model choice: `roberta-base`.** A strong, general-purpose encoder — a solid, fast baseline choice for English text classification before considering domain-specific alternatives (e.g. Legal-BERT, explored in the FFT notebook).

**Tokenization:** documents are truncated to `MAX_LEN=384` tokens, informed by the word-count distribution from Section 2. Padding is left dynamic (handled later by the data collator) rather than padding every example upfront, saving compute.


In [5]:
MODEL_NAME = "roberta-base"
MAX_LEN = 384   # covers most docs without being too slow; bump to 512 if GPU allows

print(f"[INFO] Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,   # dynamic padding via DataCollator, faster
    )

train_ds = Dataset.from_pandas(train_split[['text', 'label']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_split[['text', 'label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['id', 'text']].reset_index(drop=True))

print("[INFO] Tokenizing train set...")
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
print("[INFO] Tokenizing val set...")
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
print("[INFO] Tokenizing test set...")
test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

print("[INFO] Sample tokenized example:", train_ds[0])


[INFO] Loading tokenizer: roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

[INFO] Tokenizing val set...


Map:   0%|          | 0/5084 [00:00<?, ? examples/s]

[INFO] Tokenizing test set...


Map:   0%|          | 0/12710 [00:00<?, ? examples/s]

[INFO] Sample tokenized example: {'labels': 9, 'input_ids': [0, 347, 5061, 6905, 3063, 26747, 6597, 6034, 36, 3586, 43, 440, 361, 6617, 73, 34972, 9, 601, 392, 4482, 524, 4345, 18912, 36, 717, 3586, 43, 440, 389, 4671, 73, 6617, 13223, 10, 17756, 1475, 12, 417, 14672, 4053, 15, 5058, 9, 35797, 40744, 34864, 11, 22514, 6, 798, 50, 4174, 1941, 230, 5061, 6905, 3063, 3243, 1941, 10353, 23075, 1889, 2604, 7744, 6, 5365, 6203, 7, 5, 20704, 10584, 5, 796, 2573, 6, 5365, 6203, 7, 1080, 18912, 36, 3586, 43, 440, 41285, 73, 5607, 9, 820, 719, 7969, 15, 2591, 136, 12961, 5058, 31, 749, 45, 453, 9, 5, 796, 2573, 36, 134, 238, 36, 17, 711, 627, 3280, 18912, 17, 27, 238, 8, 11, 1989, 15219, 290, 6, 365, 1640, 246, 238, 733, 8, 820, 1640, 438, 43, 25991, 6, 5365, 6203, 7, 5, 2570, 4813, 30, 5, 1463, 71, 8623, 5, 11067, 1674, 6, 19563, 35, 83, 4, 36352, 1691, 12435, 112, 4, 34131, 11, 1370, 36, 134, 43, 870, 18912, 36, 3586, 43, 440, 361, 4563, 73, 17472, 36, 176, 43, 5, 1080, 13522, 8, 3112, 5, 1797

**Insight:** Tokenization completes quickly for ~51K combined documents. The printed sample confirms `input_ids`/`attention_mask` were generated correctly and the `labels` column was renamed as `Trainer` expects.


# 6. Load Base Model + Apply LoRA (PEFT)

This is the core PEFT step. We load the base `roberta-base` classification model, then wrap it with `get_peft_model` using a `LoraConfig`:

- **`r=16`** — the LoRA rank (size of the low-rank adapter matrices). Higher rank = more capacity, more trainable parameters.
- **`lora_alpha=32`** — scaling factor for the adapter output.
- **`target_modules=["query", "value"]`** — LoRA adapters are injected only into the attention layers' query and value projections (standard for BERT/RoBERTa-family models).
- **`bias="none"`** — bias terms are not adapted.

`model.print_trainable_parameters()` confirms only a small fraction of total parameters are actually trainable — this is the defining characteristic of PEFT, and the reason LoRA fine-tuning is dramatically faster and more memory-efficient than full fine-tuning.


In [6]:
print(f"[INFO] Loading base model: {MODEL_NAME} with {NUM_LABELS} labels")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                     # LoRA rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],   # attention q/v projections for RoBERTa/BERT-family
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()   # shows how tiny % of params actually train
model.enable_input_require_grads()   # required so gradient checkpointing works when base model params are frozen (LoRA)
model.to(DEVICE)
print("[INFO] LoRA model ready.")


[INFO] Loading base model: roberta-base with 30 labels


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,203,486 || all params: 125,872,188 || trainable%: 0.9561
[INFO] LoRA model ready.


**Insight:** Expect output like `trainable params: 1,203,486 || all params: 125,872,188 || trainable%: 0.9561` — confirming **under 1% of parameters are trainable**. This is PEFT/LoRA in action: the base model's 125M parameters stay frozen, and only ~1.2M small adapter parameters (plus the classification head) are updated during training.


# 7. Training Configuration

This cell defines:
1. **`compute_metrics`** — accuracy, computed at every evaluation.
2. **`TrainingArguments`** — notably, **`learning_rate=2e-4`**, roughly 10x higher than what full fine-tuning would use (2e-5). LoRA adapters are randomly initialized small matrices that need a larger learning rate to converge in the same number of steps as full fine-tuning would with all parameters starting from pretrained values.
3. **`gradient_checkpointing=True`** and **`group_by_length=True`** — memory and speed optimizations well-suited to this dataset's highly variable document lengths.


In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    print(f"[EVAL] Accuracy: {acc:.4f}")
    return {"accuracy": acc}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,          # LoRA usually needs a higher LR than full fine-tune
    per_device_train_batch_size=8,    # reduced to avoid OOM, especially with group_by_length long batches
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,    # keeps effective batch size = 32
    gradient_checkpointing=True,      # trades a bit of speed for much lower memory use
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="/kaggle/working/logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),   # mixed precision - big speedup on GPU
    report_to="none",
    save_total_limit=2,
    dataloader_num_workers=2,   # parallel CPU data loading, reduces GPU idle time
    group_by_length=True,       # buckets similar-length docs together -> less padding waste, big speedup for long/variable legal text
)

print(f"[INFO] Training configured for {training_args.num_train_epochs} epochs.")


[INFO] Training configured for 4 epochs.


**Insight:** `group_by_length=True` is particularly valuable here given how much this dataset's document lengths vary (from short decisions to long, detailed regulations) — it meaningfully reduces wasted padding computation.


# 8. Model Training

With an effective batch size of 32 (8 per device × 4 gradient accumulation steps), mixed precision (`fp16`), and gradient checkpointing, this LoRA fine-tuning run completes in roughly **1.5 hours on a single Kaggle T4 GPU** for 4 epochs — notably faster than the equivalent full fine-tuning run, one of PEFT's core advantages.


In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("[INFO] Starting training...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"[INFO] GPU memory before training: allocated={torch.cuda.memory_allocated()/1e9:.2f}GB, reserved={torch.cuda.memory_reserved()/1e9:.2f}GB")
train_result = trainer.train()
print(f"[INFO] Training finished. Train loss: {train_result.training_loss:.4f}")


[INFO] Starting training...
[INFO] GPU memory before training: allocated=0.50GB, reserved=0.51GB


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.095500,1.088204,0.669158
2,1.021100,1.017678,0.684107
3,0.965400,0.977840,0.694925
4,0.940400,0.958696,0.701416


[EVAL] Accuracy: 0.6692


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

[EVAL] Accuracy: 0.6949


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[EVAL] Accuracy: 0.7014


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[INFO] Training finished. Train loss: 1.0856


**Insight:** Training loss should decrease steadily across the 4 epochs. Because only the small LoRA adapters are being updated, the risk of catastrophic overfitting is lower than full fine-tuning — but the accuracy ceiling is correspondingly lower too, which is the fundamental PEFT trade-off (efficiency vs. peak accuracy).


# 9. Final Evaluation & Test Set Inference

We confirm the best checkpoint's validation accuracy, then run inference on the test set.


In [9]:
val_metrics = trainer.evaluate()
print(f"[INFO] Final validation metrics: {val_metrics}")

print("[INFO] Running inference on test set...")
test_ds_for_pred = test_ds.remove_columns(['id'])
preds_output = trainer.predict(test_ds_for_pred)
test_preds = np.argmax(preds_output.predictions, axis=-1)
print(f"[INFO] Prediction shape: {test_preds.shape}")
print(f"[INFO] Prediction label distribution:\n{pd.Series(test_preds).value_counts().sort_index()}")


[EVAL] Accuracy: 0.7014
[INFO] Final validation metrics: {'eval_loss': 0.958695650100708, 'eval_accuracy': 0.7014162077104642, 'eval_runtime': 47.2175, 'eval_samples_per_second': 107.672, 'eval_steps_per_second': 6.735, 'epoch': 4.0}
[INFO] Running inference on test set...
[INFO] Prediction shape: (12710,)
[INFO] Prediction label distribution:
0      772
1     1286
2      514
3      488
4      429
5      917
6      723
7     1724
8     1554
9      586
10     177
11     865
12     763
13     399
14     182
15     200
16     211
17      42
18      30
19     479
20      29
21       5
22      11
23      77
25     164
26      74
28       9
Name: count, dtype: int64


**Insight:** This run achieved **0.708** on the leaderboard. Compare the prediction distribution against the training label distribution (Section 2) — classes that were rare in training tend to be under-represented or entirely absent from predictions, a known limitation of this simpler LoRA setup (no class-weighting was applied here).


# 10. Submission File & Adapter Saving

We build the submission CSV in Kaggle's expected format (**uppercase `ID`**, `label`), validate row counts, and save the LoRA adapter weights (not the full model — since base weights are frozen and unchanged, only the small adapter needs to be persisted for reuse).


In [10]:
submission = pd.DataFrame({
    "ID": test_df["id"],
    "label": test_preds
})
print(submission.head())
assert submission.shape[0] == sample_sub.shape[0], "Row count mismatch with sample_submission!"

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("[INFO] submission.csv saved to /kaggle/working/submission.csv")

# Save LoRA adapter weights only (base model is unchanged, no need to save it)
model.save_pretrained("/kaggle/working/lora_adapter")
print("[INFO] LoRA adapter saved.")


   ID  label
0   0      7
1   1      2
2   2      3
3   3     11
4   4      0
[INFO] submission.csv saved to /kaggle/working/submission.csv
[INFO] LoRA adapter saved.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


**Insight:** `submission.csv` is ready to submit. Note the saved artifact is just the LoRA adapter (a few MB) rather than a full ~500MB model checkpoint — another practical PEFT benefit: adapters are cheap to store, version, and swap between tasks on top of the same frozen base model.

---

## Summary & Comparison

| Metric | This notebook (LoRA/PEFT) |
|---|---|
| Base model | roberta-base |
| Trainable parameters | ~1.2M (~1%) |
| Training time | ~1.5 hours |
| Leaderboard accuracy | **0.708** |

A separate notebook using **Full Fine-Tuning** on a domain-matched legal-BERT model reached a higher accuracy (~0.73-0.75) at similar training time, illustrating the classic PEFT trade-off: LoRA is faster to iterate with and far cheaper to store, but full fine-tuning had more headroom on this particular task.
